# 1. Data Loading and Initial Inspection

In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('yellow_tripdata_2025-01.parquet')

In [3]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

## 1.1. Feature Selection
To optimize memory usage and focus strictly on business-relevant metrics, we filter the dataset to retain only the essential columns.

In [5]:
useful = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID', 'fare_amount', 'tip_amount', 'total_amount', 'RatecodeID', 'payment_type']

In [6]:
df = df[useful]

## 1.2. Baseline Statistical Summary
We run a statistical summary to locate initial data anomalies (e.g., negative fares, unrealistic trip distances, or extreme passenger counts).

In [7]:
df.describe()

,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,tip_amount,total_amount,RatecodeID,payment_type
count,2.935077e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06,2.935077e+06,3.475226e+06
mean,1.297859e+00,5.855126e+00,1.651916e+02,1.641252e+02,1.708180e+01,2.959813e+00,2.561129e+01,2.482535e+00,1.036623e+00
std,7.507503e-01,5.646016e+02,6.452948e+01,6.940169e+01,4.634729e+02,3.779681e+00,4.636585e+02,1.163277e+01,7.013334e-01
min,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,-9.000000e+02,-8.600000e+01,-9.010000e+02,1.000000e+00,0.000000e+00
25%,1.000000e+00,9.800000e-01,1.320000e+02,1.130000e+02,8.600000e+00,0.000000e+00,1.520000e+01,1.000000e+00,1.000000e+00
50%,1.000000e+00,1.670000e+00,1.620000e+02,1.620000e+02,1.211000e+01,2.450000e+00,1.995000e+01,1.000000e+00,1.000000e+00
75%,1.000000e+00,3.100000e+00,2.340000e+02,2.340000e+02,1.950000e+01,3.930000e+00,2.778000e+01,1.000000e+00,1.000000e+00
max,9.000000e+00,2.764236e+05,2.650000e+02,2.650000e+02,8.633721e+05,4.000000e+02,8.633804e+05,9.900000e+01,5.000000e+00


## 1.3. Deep Dive into Anomalies
Before applying filters, we investigate specific distributions to understand the nature of the data corruption (e.g., system glitches vs. extreme real-world events).

In [8]:
# Checking the distribution of passenger counts to find empty or overloaded trips
df['passenger_count'].value_counts()

1.0    2322434
2.0     407761
3.0      91409
4.0      59009
0.0      24656
5.0      17786
6.0      12004
8.0         11
7.0          4
9.0          3
Name: passenger_count, dtype: int64

In [9]:
# Analyzing extreme trip distances using high percentiles
print("Trip Distance Quantiles:")
print(df['trip_distance'].quantile([0.95, 0.99, 0.999, 0.9999]))

Trip Distance Quantiles:
0.9500    11.830000
0.9900    19.500000
0.9990    28.857750
0.9999    58.424775
Name: trip_distance, dtype: float64


In [10]:
# Investigating financial metrics for trips with 0.0 recorded distance
print("Fare distribution for zero-distance trips:")
print(df[df['trip_distance'] == 0]['fare_amount'].describe())

Fare distribution for zero-distance trips:
count    90893.000000
mean        17.657909
std         31.187945
min       -850.000000
25%          3.000000
50%         13.310000
75%         22.730000
max        950.000000
Name: fare_amount, dtype: float64


### 1.3.1. Payment Type Bias Analysis
We analyze the relationship between payment types and recorded tip amounts. In the NYC Taxi dataset, cash transactions (Type 2) typically do not record tips in the system, creating a heavy downward bias.

In [11]:

print("Average tip by payment type:")
print(df.groupby('payment_type')['tip_amount'].mean())

Average tip by payment type:
payment_type
0    0.452353
1    4.106073
2    0.002714
3    0.016866
4    0.043985
5    0.000000
Name: tip_amount, dtype: float64


> **Conclusion:** Since cash trips (Type 2) show an average tip of nearly $0.0$, using the standard mean (`mean`) for final aggregation would skew our insights. To eliminate this bias and capture true passenger behavior, we must use the **median** in our final analysis.

# 2. Data Cleaning & Noise Filtering
In this section, we apply a multi-stage filtering pipeline based on our previous statistical findings to remove corrupted data and isolate a clean analytical dataset.

## 2.1. Defining Filtering Conditions
We construct a unified boolean mask to filter out telemetry glitches, extreme outliers, and logically impossible records (e.g., negative duration, impossible fares).

In [12]:
# Calculate trip duration in minutes for validation
duration_min = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Define logic masks
mask_passengers = df['passenger_count'].between(1, 6)
mask_distance   = (df['trip_distance'] > 0) & (df['trip_distance'] <= 60)
mask_duration   = duration_min.between(1, 120)
mask_fare       = df['fare_amount'].between(2.5, 200)

# Special rule for traffic gridlock (zero distance but valid fare)
mask_zero_dist_traffic = (df['trip_distance'] == 0) & df['fare_amount'].between(2.5, 70)

# Combine masks and overwrite df using .copy() to avoid SettingWithCopyWarning
df = df[mask_passengers & mask_duration & (mask_distance | mask_zero_dist_traffic) & mask_fare].copy()

In [13]:
print(len(df))

2816316


# 3. Feature Engineering & Business Metrics Calculation
In this section, we extract temporal components (hour and day of the week) from the pickup timestamp and calculate the relative tip ratio to analyze passenger tipping behavior.

In [14]:
# Extracting temporal features
df['hour'] = df['tpep_pickup_datetime'].dt.hour
df['day'] = df['tpep_pickup_datetime'].dt.dayofweek

# Calculating relative tip ratio (%) safely
df['tip_ratio'] = (df['tip_amount'] / df['fare_amount'] * 100).where(df['fare_amount'] > 0, 0).round(1)

# 4. Data Aggregation & Business Insights

## 4.1. Building the Hourly Tip Ratio Matrix
We construct a pivot table to analyze the median tip ratio across all hours of the day for each day of the week (0 = Monday, 6 = Sunday).

In [15]:
# Generating the 7x24 matrix using median to eliminate cash-trip biases
tip_matrix = df.pivot_table(
    index='day', 
    columns='hour', 
    values='tip_ratio', 
    aggfunc='median'
)

# Displaying the raw matrix
tip_matrix

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
day,,,,,,,,,,,,,,,,,,,,,
0,23.1,22.6,21.9,20.65,19.6,16.90,22.40,23.6,24.7,25.2,...,25.0,25.1,26.8,27.6,27.9,27.6,26.8,26.5,25.6,23.4
1,23.1,23.1,21.2,14.40,13.4,20.00,23.10,24.6,25.0,25.0,...,25.1,25.2,27.8,28.5,28.9,28.9,27.1,27.1,26.7,25.9
2,24.7,24.6,24.6,24.30,23.2,20.80,23.10,24.5,25.2,25.2,...,24.7,24.8,26.8,27.6,28.0,27.9,26.8,26.7,26.5,25.6
3,24.7,24.9,24.9,21.60,18.3,20.25,21.90,24.3,24.9,24.8,...,24.6,24.6,26.9,27.8,27.9,28.3,26.7,26.5,26.2,25.9
4,25.2,25.4,24.6,23.00,22.4,20.80,21.40,23.6,24.8,24.8,...,24.6,24.6,26.7,27.8,27.9,27.8,26.1,26.2,26.0,26.0
5,26.2,26.5,26.1,25.80,23.7,18.70,20.10,21.5,24.0,25.2,...,24.9,24.8,24.6,24.8,24.9,24.9,25.6,25.9,25.8,25.8
6,26.2,26.2,26.3,25.30,24.5,20.00,19.25,22.4,24.2,25.2,...,25.1,24.9,24.9,25.2,25.4,25.3,26.0,26.0,24.9,23.6


In [16]:
# Turning the boring 7x24 table into an instant visual heatmap
tip_matrix.style.background_gradient(cmap='YlOrRd', axis=None).format("{:.1f}%")

hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
day,,,,,,,,,,,,,,,,,,,,,,,,
0,23.1%,22.6%,21.9%,20.6%,19.6%,16.9%,22.4%,23.6%,24.7%,25.2%,25.2%,25.2%,25.3%,25.4%,25.0%,25.1%,26.8%,27.6%,27.9%,27.6%,26.8%,26.5%,25.6%,23.4%
1,23.1%,23.1%,21.2%,14.4%,13.4%,20.0%,23.1%,24.6%,25.0%,25.0%,25.0%,24.9%,25.0%,25.1%,25.1%,25.2%,27.8%,28.5%,28.9%,28.9%,27.1%,27.1%,26.7%,25.9%
2,24.7%,24.6%,24.6%,24.3%,23.2%,20.8%,23.1%,24.5%,25.2%,25.2%,24.9%,24.8%,24.8%,24.8%,24.7%,24.8%,26.8%,27.6%,28.0%,27.9%,26.8%,26.7%,26.5%,25.6%
3,24.7%,24.9%,24.9%,21.6%,18.3%,20.2%,21.9%,24.3%,24.9%,24.8%,24.7%,24.5%,24.6%,24.6%,24.6%,24.6%,26.9%,27.8%,27.9%,28.3%,26.7%,26.5%,26.2%,25.9%
4,25.2%,25.4%,24.6%,23.0%,22.4%,20.8%,21.4%,23.6%,24.8%,24.8%,24.8%,24.6%,24.7%,24.6%,24.6%,24.6%,26.7%,27.8%,27.9%,27.8%,26.1%,26.2%,26.0%,26.0%
5,26.2%,26.5%,26.1%,25.8%,23.7%,18.7%,20.1%,21.5%,24.0%,25.2%,25.4%,25.4%,25.1%,24.9%,24.9%,24.8%,24.6%,24.8%,24.9%,24.9%,25.6%,25.9%,25.8%,25.8%
6,26.2%,26.2%,26.3%,25.3%,24.5%,20.0%,19.2%,22.4%,24.2%,25.2%,25.6%,25.8%,25.6%,25.3%,25.1%,24.9%,24.9%,25.2%,25.4%,25.3%,26.0%,26.0%,24.9%,23.6%
